# Allstate Claims Severity
## 02 — Preprocessing and Baseline Modeling

### Objective
Build a reproducible validation framework and establish a simple baseline for predicting insurance claim severity.

### In This Notebook
- Load the labeled training dataset.
- Separate predictive features from the target variable.
- Create a reproducible train-validation split.
- Establish a baseline using Mean Absolute Error (MAE).
- Prepare for leakage-safe feature preprocessing.

We begin with a simple holdout split. Model fitting and preprocessing will be introduced step by step.

## 1. Data Preparation

### 1.1 Data Loading

We load the labeled training data using the same project structure as in the previous notebook.

The separate competition test dataset is reserved for final predictions and will not be used for validation.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

data_dir = project_root / "data" / "raw"
train = pd.read_csv(data_dir / "train.csv")

print("Train shape:", train.shape)

Train shape: (188318, 132)


### 1.2 Feature and Target Separation

The target `loss` contains the claim amount we want to predict. We separate it from the input features to prevent target leakage.

The `id` column is a record identifier rather than a claim characteristic, so it is excluded from the initial feature set.

In [2]:
X = train.drop(columns=["id", "loss"])
y = train["loss"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

Feature matrix shape: (188318, 130)
Target shape: (188318,)


## 2. Train–Validation Split

To estimate how the model performs on unseen labeled records, we split the available training data into two parts:

- **Training set (80%):** Used to learn model parameters and fit preprocessing steps.
- **Validation set (20%):** Used to evaluate predictive performance on held-out data.

A fixed `random_state` makes the random split reproducible. No preprocessing is fitted before the split.

In [3]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("y_train:", y_train.shape)
print("y_valid:", y_valid.shape)

X_train: (150654, 130)
X_valid: (37664, 130)
y_train: (150654,)
y_valid: (37664,)


### 2.1 Split Sanity Checks

Before modeling, we verify that the split preserves all observations without placing the same record in both subsets.

In [4]:
print("Total records:", len(X_train) + len(X_valid))
print("Original records:", len(X))
print("Overlapping indices:", len(X_train.index.intersection(X_valid.index)))

Total records: 188318
Original records: 188318
Overlapping indices: 0



## 3. Baseline Modeling

Before training predictive models, we establish a simple
baseline to evaluate whether more complex approaches
provide meaningful improvements.

We use a `DummyRegressor` with the median strategy,
which predicts the median training loss for every record.

This approach is appropriate for our initial benchmark
because the target distribution is strongly right-skewed
and the evaluation metric is Mean Absolute Error (MAE).

The baseline will serve as a reference for evaluating
future models on the same validation set.


In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error

dummy_model = DummyRegressor(strategy="median")

dummy_model.fit(X_train, y_train)

,"strategy strategy: {""mean"", ""median"", ""quantile"", ""constant""}, default=""mean""Strategy to use to generate predictions.* ""mean"": always predicts the mean of the training set* ""median"": always predicts the median of the training set* ""quantile"": always predicts a specified quantile of the training set, provided with the quantile parameter.* ""constant"": always predicts a constant value that is provided by the user.",'median'
,"constant constant: int or float or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
,"quantile quantile: float in [0.0, 1.0], default=NoneThe quantile to predict using the ""quantile"" strategy. A quantile of0.5 corresponds to the median, while 0.0 to the minimum and 1.0 to themaximum.",None
Name,Type,Value
"constant_ constant_: ndarray of shape (1, n_outputs)Mean or median or quantile of the training targets or constant valuegiven by the user.","ndarray[float64](1, 1)",[[2115.89]]
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X` hasfeature names that are all strings.","ndarray[object](130,)","['cat1','cat2','cat3',...,'cont12','cont13','cont14']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`.,int,130
n_outputs_ n_outputs_: intNumber of outputs.,int,1


In [6]:
dummy_pred = dummy_model.predict(X_valid)

dummy_mae = mean_absolute_error(y_valid, dummy_pred)


print(f"Training median loss: {y_train.median():,.2f}")
print(f"Dummy baseline MAE: {dummy_mae:,.2f}")

Training median loss: 2,115.89
Dummy baseline MAE: 1,796.96


In [7]:
print("First five predictions:", dummy_pred[:5])
print("Training median:", y_train.median())

First five predictions: [2115.895 2115.895 2115.895 2115.895 2115.895]
Training median: 2115.895



### 3.1 Baseline Findings

The median-based DummyRegressor achieved a validation MAE
of **1,796.96**, using a constant prediction of 2,115.895
for every validation record.

**Key Findings:**
- The model does not use feature information to make predictions.
- The training median provides a simple reference for evaluating future models.
- The baseline establishes an initial performance benchmark on the validation set.

**Modeling Decision:**

All subsequent models will be evaluated using the same
validation split and MAE metric. A useful predictive model
should improve upon this baseline by learning relationships
between the input features and claim severity.



## 4. Feature Preprocessing

The dataset contains both categorical and continuous
features, which require different preprocessing approaches.

Categorical features must be encoded into a numerical
representation suitable for machine learning models,
while continuous features can be processed according
to the requirements of the selected estimator.

To prevent data leakage, preprocessing transformations
will be fitted exclusively on the training subset and
then applied to the validation subset.

We first identify feature types before constructing
a preprocessing pipeline.


In [10]:
cat_cols = X_train.select_dtypes(include=["object", "str", "category"]).columns.tolist()

num_cols = X_train.select_dtypes(include="number").columns.tolist()


print("Categorical features:", len(cat_cols))
print("Numerical features:", len(num_cols))
print("Total selected features:", len(cat_cols) + len(num_cols))
print("Original features:", X_train.shape[1])

Categorical features: 116
Numerical features: 14
Total selected features: 130
Original features: 130


In [11]:
print(X_train[cat_cols[:5]].dtypes)
print(X_train[num_cols[:5]].dtypes)

cat1    str
cat2    str
cat3    str
cat4    str
cat5    str
dtype: object
cont1    float64
cont2    float64
cont3    float64
cont4    float64
cont5    float64
dtype: object



### 4.1 Feature Type Findings

All 130 input features were successfully identified:
116 categorical and 14 numerical features.

Categorical features use string-based data types, while
continuous features are stored as floating-point values.

**Preprocessing Decision:**
- Apply One-Hot Encoding to categorical features.
- Preserve numerical features initially.
- Fit preprocessing transformations on the training subset only.

This approach retains all original predictors while
preparing the data for the first feature-based model.



### 4.3 Preprocessing with ColumnTransformer

We use `ColumnTransformer` to apply different preprocessing
operations to categorical and numerical features.

- Categorical features are encoded using One-Hot Encoding.
- Numerical features are retained without transformation.
- Unknown categories are handled without raising errors.
- Preprocessing is fitted only on the training subset.

This approach provides a consistent feature representation
for both training and validation data.


In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ]
)

In [15]:
X_train_processed = preprocessor.fit_transform(X_train)

X_valid_processed = preprocessor.transform(X_valid)

print("Original training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_valid_processed.shape)

Original training shape: (150654, 130)
Processed training shape: (150654, 1130)
Processed validation shape: (37664, 1130)


In [16]:
print("Output is sparse:", preprocessor.sparse_output_)
print(
    "Same number of features:", X_train_processed.shape[1] == X_valid_processed.shape[1]
)

print("First 10 feature names:", preprocessor.get_feature_names_out()[:10])

Output is sparse: True
Same number of features: True
First 10 feature names: ['cat__cat1_A' 'cat__cat1_B' 'cat__cat2_A' 'cat__cat2_B' 'cat__cat3_A'
 'cat__cat3_B' 'cat__cat4_A' 'cat__cat4_B' 'cat__cat5_A' 'cat__cat5_B']



### 4.3 Preprocessing Findings

The preprocessing pipeline expanded the original 130 features
into 1,130 model-ready features.

**Key Findings:**
- One-Hot Encoding generated 1,116 categorical features.
- All 14 numerical features were preserved without transformation.
- Both training and validation data contain 1,130 features.
- The resulting feature matrices use a sparse representation.

**Modeling Decision:**

The transformed features will be used to establish the first
feature-based regression model.

Preprocessing will be integrated into a scikit-learn Pipeline
to ensure consistent transformations during training
and prediction.



## 5. Ridge Regression Baseline

We train a Ridge Regression model as our first
feature-based baseline.

Ridge applies L2 regularization to a linear regression
model, helping control coefficient magnitudes.

A scikit-learn Pipeline combines preprocessing and
regression to ensure consistent transformations.

Model performance will be evaluated using validation
MAE and compared against the median-based DummyRegressor.


In [18]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

ridge_pipeline = Pipeline(
    steps=[("preprocessing", preprocessor), ("model", Ridge(alpha=1.0, solver="lsqr"))]
)

In [19]:
ridge_pipeline.fit(X_train, y_train)

ridge_pred = ridge_pipeline.predict(X_valid)

ridge_mae = mean_absolute_error(y_valid, ridge_pred)

print(f"Dummy baseline MAE: {dummy_mae:,.2f}")
print(f"Ridge validation MAE: {ridge_mae:,.2f}")
print(f"MAE improvement: {dummy_mae - ridge_mae:,.2f}")

Dummy baseline MAE: 1,796.96
Ridge validation MAE: 1,290.79
MAE improvement: 506.17



### 5.1 Ridge Regression Findings

Ridge Regression achieved a validation MAE of **1,290.79**,
compared with **1,796.96** for the median-based baseline.

This represents an absolute MAE reduction of 506.17
(approximately 28.17%).

**Key Findings:**
- Ridge substantially outperformed the constant-prediction baseline.
- The encoded features contain useful predictive information.
- Regularized linear regression provides a stronger benchmark for future models.

**Next Step:**

Compare training and validation errors to assess
generalization before proceeding to more complex models.


In [20]:
ridge_train_pred = ridge_pipeline.predict(X_train)

ridge_train_mae = mean_absolute_error(y_train, ridge_train_pred)

print(f"Training MAE: {ridge_train_mae:,.2f}")
print(f"Validation MAE: {ridge_mae:,.2f}")
print(f"MAE gap: {ridge_mae - ridge_train_mae:,.2f}")

Training MAE: 1,293.28
Validation MAE: 1,290.79
MAE gap: -2.49



### 5.2 Generalization Assessment

The Ridge model achieved a training MAE of **1,293.28**
and a validation MAE of **1,290.79**, resulting in a
generalization gap of **-2.49**.

**Key Findings:**
- Training and validation errors are nearly identical.
- No substantial overfitting is evident from the MAE comparison.
- Ridge provides a stable initial feature-based benchmark.

**Modeling Decision:**

Retain Ridge as the current benchmark. Before comparing
more complex models, inspect the prediction range and
check for potentially invalid predictions.


In [22]:
negative_mask = ridge_pred < 0

print("Negative predictions:", negative_mask.sum())
print(f"Negative prediction rate: {negative_mask.mean():.2%}")

print(f"Minimum prediction: {ridge_pred.min():,.2f}")
print(f"Maximum prediction: {ridge_pred.max():,.2f}")

Negative predictions: 324
Negative prediction rate: 0.86%
Minimum prediction: -1,527.86
Maximum prediction: 35,958.33



### 5.3 Prediction Range Assessment

The Ridge model generated 324 negative predictions,
representing 0.86% of the validation set.

Predicted values ranged from -1,527.86 to 35,958.33.

Although negative predictions affect a small proportion
of records, they are inconsistent with the non-negative
nature of claim severity.

**Next Step:**

Evaluate non-negative prediction clipping and measure
its impact on validation MAE before adopting it as a
post-processing strategy.


In [24]:
import numpy as np

ridge_pred_clipped = np.clip(ridge_pred, a_min=0, a_max=None)

ridge_clipped_mae = mean_absolute_error(y_valid, ridge_pred_clipped)

print(f"Original Ridge MAE: {ridge_mae:,.2f}")
print(f"Clipped Ridge MAE: {ridge_clipped_mae:,.2f}")
print(f"MAE improvement: {ridge_mae - ridge_clipped_mae:,.2f}")

Original Ridge MAE: 1,290.79
Clipped Ridge MAE: 1,288.44
MAE improvement: 2.36



### 5.4 Prediction Clipping Findings

Clipping negative Ridge predictions to zero reduced
validation MAE from **1,290.79** to **1,288.44**.

This represents a small improvement of approximately
0.18% relative to the original Ridge model.

**Key Findings:**
- The original Ridge model produced 324 negative predictions (0.86%).
- Clipping removed these invalid predictions.
- The improvement in overall MAE was modest due to the small proportion of affected records.

**Modeling Decision:**

Retain non-negative prediction clipping as a
post-processing step for Ridge Regression.

The clipped Ridge model will serve as the current
benchmark for future model comparisons.


In [25]:
results = pd.DataFrame(
    {
        "model": ["Dummy Regressor", "Ridge Regression", "Ridge + Clipping"],
        "validation_mae": [dummy_mae, ridge_mae, ridge_clipped_mae],
    }
)

results = results.sort_values("validation_mae", ascending=True)

display(results.round(2))

,model,validation_mae
2,Ridge + Clipping,1288.44
1,Ridge Regression,1290.79
0,Dummy Regressor,1796.96
